# Import

## Lib

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pandas as pd
from tqdm import tqdm

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

## Data

In [ ]:
data_path = 'Data/Fairness_tabular_datasets/'

In [3]:
os.listdir('Dataset/Fairness/')

['ACSEmploymentFiltered_2023.csv',
 'ACSMobility_2023.csv',
 'ACSPublicCoverage_2023.csv',
 'ACSTravelTime_2023.csv',
 'ASCIncome_2023.csv',
 'bar_pass_prediction.csv',
 'Compas_dataset.csv',
 'german.data-numeric',
 'german_credit_data.csv',
 'statlog+german+credit+data',
 'UCI_Credit_Card.csv']

In [ ]:
df_ASC_Employment = pd.read_csv(data_path + 'ACSEmploymentFiltered_2023.csv')
df_ASC_Mobility = pd.read_csv(data_path + 'ACSMobility_2023.csv')
df_ASC_PublicCoverage = pd.read_csv(data_path + 'ACSPublicCoverage_2023.csv')
df_ASC_TravelTime = pd.read_csv(data_path + 'ACSTravelTime_2023.csv')
df_ASC_Income = pd.read_csv(data_path + 'ASCIncome_2023.csv')
df_bar_pass = pd.read_csv(data_path + 'bar_pass_prediction.csv')
df_Compas = pd.read_csv(data_path + 'Compas_dataset.csv')
df_German_credit = pd.read_csv(data_path + 'german_credit_data.csv')
df_UCI_Credit = pd.read_csv(data_path + 'UCI_Credit_Card.csv')

dic_dfs = {
    'ASC_EMP': df_ASC_Employment,
    'ASC_MOB': df_ASC_Mobility,
    'ASC_PUC': df_ASC_PublicCoverage,
    'ASC_TRA': df_ASC_TravelTime,
    'ASC_INC': df_ASC_Income,
    'BAR_PASS': df_bar_pass,
    'COMPAS': df_Compas,
    'UCI_CREDIT': df_UCI_Credit,
}

We define below for each dataset:
- the target name: the name of the variable which will be predicted: Y
- the sensitive variable S (on which fairness should be evaluated)
- threshold: target variable can be at first binary $(Y \in \{0,1\})$ (and nothing have to be done) or discret with multiple values ($Y \in 0, ..., N$), or continuous ($Y \in \mathbb{R}$). We only consider binary classification problem in this setting, therefore we transform the problem into one. For instance, in the Income dataset, the regression problem (how much is the individual earning?) to a binary classification problem (is the individual earning more than threshold?).
- S_fct: we transform the sensitive variable, after the transformation, $S \in \{0,1\}$. An example would be considering in the Mobility dataset person over and under 25 years old.
- column_to_drop: Column which won't be used to do the predictions.
- column_to_dommy: column which have categorical data transformed into numerical which we one-hot encode for better performance (and meaning).

In [5]:
dict_dfs_variable_names = {
    'ASC_EMP' : {'S_variable_name' : 'DIS',
                 'target_name' : 'ESR',
                 'threshold' : 0.5,
                 'S_fct' : lambda S : S - 1,
                 'column_to_drop' : ['RAC1P', 'State', 'RAC1P.1'],
                 'column_to_dummy' : ['MAR', 'CIT', 'MIL', 'ANC', 'GCL', 'MIG', 'ESP'],
                },
    'ASC_INC' : {'S_variable_name' : 'SEX',
                 'target_name' : 'PINCP',
                 'threshold' : 50000,
                 'S_fct' : lambda S : 2 - S,
                 'column_to_drop' : ['RAC1P', 'State', 'RAC1P.1'],
                 'column_to_dummy' : ['MAR', 'COW'],
                },
    'ASC_PUC' : {'S_variable_name' : 'DIS',
                 'target_name' : 'PUBCOV',
                 'threshold' : 0.5,
                 'S_fct' : lambda S : 2 - S,
                 'column_to_drop' : ['RAC1P', 'State', 'RAC1P.1'],
                 'column_to_dummy' : ['MAR', 'CIT', 'MIL', 'ANC', 'ESR', 'MIG', 'ESP'],
                },
    'ASC_TRA' : {'S_variable_name' : 'SEX',
                 'target_name' : 'JWMNP',
                 'threshold' : 20,
                 'S_fct' : lambda S : 2 - S,
                 'column_to_drop' : ['RAC1P', 'State', 'RAC1P.1'],
                 'column_to_dummy' : ['MAR', 'CIT', 'MIL', 'ANC', 'GCL', 'COW', 'ESR', 'MIG', 'ESP'],
                },
    'ASC_MOB' : {'S_variable_name' : 'AGEP',
                 'target_name' : 'MIG',
                 'threshold' : 0.5,
                 'S_fct' : lambda S : S > 25,
                 'column_to_drop' : ['RAC1P', 'State', 'RAC1P.1'],
                 'column_to_dummy' : ['MAR', 'CIT', 'MIL', 'ANC', 'GCL', 'COW', 'ESR'],
                },
    'BAR_PASS' : {'S_variable_name' : 'sex',
                  'target_name' : 'pass_bar',
                  'threshold' : 0.5,
                  'S_fct' : lambda S : S - 1,
                  'column_to_drop' : ['ID', 'race', 'bar', 'index6040', 'indxgrp', 'indxgrp2', 'dnn_bar_pass_prediction', 'bar1', 'bar1_yr', 'bar2', 'bar2_yr', 'gpa', 'zgpa', 'zfygpa', 'cluster', 'other', 'asian', 'black', 'hisp', 'bar_passed', 'gender', 'male', 'race1', 'race2', 'Dropout', 'parttime', 'grad'],
                  'column_to_dummy' : [],
    },
    'COMPAS' : {'S_variable_name' : 'sex',
                'target_name' : 'is_violent_recid',
                'threshold' : 0.5,
                'S_fct' : lambda S : S,
                'column_to_drop' : ['is_recid', 'score_text', 'race_African-American', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Native American', 'race_Other'],
                'column_to_dummy' : [],
    },
    'UCI_CREDIT' : {'S_variable_name' : 'SEX',
                    'target_name' : 'default.payment.next.month',
                    'threshold' : 0.5,
                    'S_fct' : lambda S : S - 1,
                    'column_to_drop' : ['ID'],
                    'column_to_dummy' : [],
    },
}

# Functions

In [ ]:
def Disparate_impact_add(S, Y, P):
    '''
    DI = |P(Ŷ=1|S=1) - P(Ŷ=1|S=0)|
    '''
    S1_index = np.where(S == 1)[0]
    S0_index = np.where(S == 0)[0]
    Proba_P_S1 = P[S1_index].mean()
    Proba_P_S0 = P[S0_index].mean()
    return abs(Proba_P_S1 - Proba_P_S0)

def Disparate_impact_mult(S, Y, P):
    '''
    DI = min(P(Ŷ=1|S=1), P(Ŷ=1|S=0)) / max(P(Ŷ=1|S=1), P(Ŷ=1|S=0))
    '''
    S1_index = np.where(S == 1)[0]
    S0_index = np.where(S == 0)[0]
    Proba_P_S1 = P[S1_index].mean()
    Proba_P_S0 = P[S0_index].mean()
    if Proba_P_S1 == 0 or Proba_P_S0 == 0:
        return 1
    return min(Proba_P_S1, Proba_P_S0) / max(Proba_P_S1, Proba_P_S0)
    
def Equality_of_Opportunity_add(S, Y, P):
    """
    EoO = |P(Ŷ=1|S=1 ∧ Y=1) - P(Ŷ=1|S=0 ∧ Y=1)|
    """
    S1Y1_index = (S == 1) & (Y == 1)
    S0Y1_index = (S == 0) & (Y == 1)
    Proba_P_Y1S1 = P[S1Y1_index].mean()
    Proba_P_Y1S0 = P[S0Y1_index].mean()
    return abs(Proba_P_Y1S1 - Proba_P_Y1S0)

def Equality_of_Opportunity_mult(S, Y, P):
    """
    EoO = min(P(Ŷ=1|S=1 ∧ Y=1) -/ P(Ŷ=1|S=0 ∧ Y=1), P(Ŷ=1|S=0 ∧ Y=1) -/ P(Ŷ=1|S=1 ∧ Y=1))
    """
    S1Y1_index = (S == 1) & (Y == 1)
    S0Y1_index = (S == 0) & (Y == 1)
    Proba_P_Y1S1 = P[S1Y1_index].mean()
    Proba_P_Y1S0 = P[S0Y1_index].mean()
    if Proba_P_Y1S1 == 0 or Proba_P_Y1S0 == 0:
        return 0
    return min(Proba_P_Y1S1, Proba_P_Y1S0) / max(Proba_P_Y1S1, Proba_P_Y1S0)

def RD_binary(Y, P, boolean_normalized = False, verbose = False):
    """
    Based on the recall definition, we compute the divergence from the average recall of the two groups.
    """
    Y0_index = np.where(Y == 0)[0]
    Y1_index = np.where(Y == 1)[0]
    Recall_Y0 = (P[Y0_index] == 0).mean()
    Recall_Y1 = (P[Y1_index] == 1).mean()
    if (Recall_Y0 == 0 and Recall_Y1 == 0):
        return 0
    if (Recall_Y0 == 0 or Recall_Y1 == 0):
        return float('inf')
    if verbose:
        print(f"Recall Y=0 : {Recall_Y0}")
        print(f"Recall Y=1 : {Recall_Y1}")
    Recall_mean = (Recall_Y0 + Recall_Y1) / 2
    if Recall_mean == 0:
        return 0
    RD = abs(Recall_Y0 - Recall_mean) / 2
    if boolean_normalized:
        RD = RD / Recall_mean
    return RD

# Predictions and Fairness analysis

## Learning process and computing fairness metrics

In [15]:
verbose_bool = False
dic_fairness_metrics = {
    'Acc' : lambda S, Y, P : accuracy_score(y_true = Y, y_pred = P),
    'DI_add' : Disparate_impact_add,
    'DI_mult' : Disparate_impact_mult,
    'EoO_add' : Equality_of_Opportunity_add,
    'EoO_mult' : Equality_of_Opportunity_mult,
    'RD_binary' : lambda S, Y, P : RD_binary(Y, P, boolean_normalized = False, verbose = verbose_bool),
    'RD_normalized_binary' : lambda S, Y, P : RD_binary(Y, P, boolean_normalized = True, verbose = verbose_bool),
}

In [ ]:

dic_metrics_dfs = {}
scaler = StandardScaler()
#clf = GradientBoostingClassifier(learning_rate=0.005, n_estimators=200, min_samples_split=50, min_weight_fraction_leaf=0.02, max_depth=6, random_state=42, max_features=15, validation_fraction=0.1, n_iter_no_change=10, tol=0.001)

for df_name, df_dic in tqdm(dict_dfs_variable_names.items()):
    clf = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, warm_start=True)

    if verbose_bool:
        print(f"Processing {df_name}...")
    df = dic_dfs[df_name]
    df = df.drop(columns = df_dic['column_to_drop'], errors = 'ignore')
    df = df.fillna(df.median())
    S = df_dic['S_fct'](df[df_dic['S_variable_name']])

    Y = df[df_dic['target_name']] > df_dic['threshold']
    if verbose_bool:
        print(f'Y mean: {Y.mean()}')
    dic_metrics_dfs[(df_name, 'Y mean')] = Y.mean()
    X = df.drop(columns=[df_dic['S_variable_name'], df_dic['target_name']])
    X = pd.get_dummies(X, columns=df_dic['column_to_dummy'])

    X_train, X_test, Y_train, Y_test, S_train, S_test = train_test_split(X, Y, S, test_size=0.2, random_state=42)
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf.fit(X_train_scaled, Y_train)
    Y_pred = clf.predict(X_test_scaled)
    dic_metrics_dfs[(df_name, 'Y pred mean')] = Y_pred.mean()
    
    ''' Y_pred mean for each group '''
    dic_metrics_dfs[(df_name, 'Y pred S0 mean')] = Y_pred[S_test == 0].mean()
    dic_metrics_dfs[(df_name, 'Y pred S1 mean')] = Y_pred[S_test == 1].mean()

    '''Y_pred mean for each S group and Y=1'''
    Y0_index = np.where(Y_test == 0)[0]
    Y1_index = np.where(Y_test == 1)[0]
    Recall_Y0 = (Y_pred[Y0_index] == 0).mean()
    Recall_Y1 = (Y_pred[Y1_index] == 1).mean()

    dic_metrics_dfs[(df_name, 'Recall Y0')] = Recall_Y0
    dic_metrics_dfs[(df_name, 'Recall Y1')] = Recall_Y1

    for metric_name, metric_fct in dic_fairness_metrics.items():
        print(f"Computing {metric_name} for {df_name}...")
        metric_value = metric_fct(S_test, Y_test, Y_pred)
        dic_metrics_dfs[(df_name, metric_name)] = metric_value

## Results

In [ ]:
pd.DataFrame(dic_metrics_dfs)

,ASC_EMP,ASC_INC,ASC_PUC,ASC_TRA,ASC_MOB,BAR_PASS,COMPAS,UCI_CREDIT
Y mean,0.555768,0.466572,0.338400,0.28305,0.888410,0.947829,0.073197,0.221200
Y pred mean,0.667634,0.440201,0.129367,0.00000,0.999980,0.994645,0.045094,0.115167
Y pred S0 mean,0.267464,0.416921,0.044309,0.00000,0.999802,0.993323,0.030211,0.118869
Y pred S1 mean,0.753877,0.461709,0.523512,0.00000,1.000000,0.995661,0.048382,0.112688
Y pred S0 Y1 mean,0.603837,0.801674,0.953008,1.00000,0.000164,0.053097,1.000000,0.951782
Y pred S1 Y1 mean,0.884603,0.716565,0.290783,0.00000,0.999998,0.997180,0.604396,0.354151
Acc,0.759884,0.761952,0.729247,0.71776,0.888528,0.949576,0.970484,0.821000
DI_add,0.486412,0.044788,0.479204,0.00000,0.000198,0.002338,0.018170,0.006182
DI_mult,0.354785,0.902995,0.084637,1.00000,0.999802,0.997652,0.624440,0.947996
EoO_add,0.363638,0.043539,0.552150,0.00000,0.000025,0.000758,0.045988,0.016579


In [ ]:
pd.DataFrame(dic_metrics_dfs).round(2)

,ASC_EMP,ASC_INC,ASC_PUC,ASC_TRA,ASC_MOB,BAR_PASS,COMPAS,UCI_CREDIT
Y mean,0.56,0.47,0.34,0.28,0.89,0.95,0.07,0.22
Y pred mean,0.67,0.44,0.13,0.00,1.00,0.99,0.05,0.12
Y pred S0 mean,0.27,0.42,0.04,0.00,1.00,0.99,0.03,0.12
Y pred S1 mean,0.75,0.46,0.52,0.00,1.00,1.00,0.05,0.11
Y pred S0 Y1 mean,0.60,0.80,0.95,1.00,0.00,0.05,1.00,0.95
Y pred S1 Y1 mean,0.88,0.72,0.29,0.00,1.00,1.00,0.60,0.35
Acc,0.76,0.76,0.73,0.72,0.89,0.95,0.97,0.82
DI_add,0.49,0.04,0.48,0.00,0.00,0.00,0.02,0.01
DI_mult,0.35,0.90,0.08,1.00,1.00,1.00,0.62,0.95
EoO_add,0.36,0.04,0.55,0.00,0.00,0.00,0.05,0.02
